# Franka：理想数据辨识

用 `panda_arm.xml` 的无夹爪 7 轴模型演示数值回归、激励搜索、基参数辨识和 `M/c/g` 分解。末帧对应 `link7`，MDH 末项为 `d=0`。

- 输入：仓库内的 MDH 参数、去除电机惯量/阻尼/接触/限位约束后的 MJCF，以及固定随机种子。
- 输出：70 个刚体参数到 43 个基参数的映射、MuJoCo 对拍误差、力矩白噪声下的辨识误差；本页不写实验文件。
- 顺序：从上到下运行。第 3 节符号对照默认跳过；完整符号导出见 [modeling.ipynb](modeling.ipynb)。

依赖：在仓库根目录执行 `pip install -e ".[sim,ident,viz,dev]"`。
README 的 62 基参数结果包含摩擦和电机惯量，来自 `robot-identify --robot franka` 的闭环实验；与本页的理想数据示例分开使用。


In [ ]:
import sys
import time
from pathlib import Path


def _project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "robot_model" / "__init__.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("找不到含 pyproject.toml 和 src/robot_model 的项目根目录")


_ROOT = _project_root(Path.cwd().resolve())
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import mujoco
import numpy as np
import sympy
from IPython.display import display

from robot_model import Robot
from robot_model.data import add_noise, sample_trajectory
from robot_model.excitation import search_excitation
from robot_model.identification import (
    DynamicsDecomposer,
    identify,
    parameter_error,
)
from robot_model.robots.franka import (
    PANDA_MDH_BODY_NAMES,
    PANDA_MDH_PARMS,
    panda_xml_path,
)
from robot_model.simulation.reference import MujocoReference, strip_to_rigid_body

print("project root:", _ROOT)
print("xml (no gripper):", panda_xml_path())


## 1. 模型与正运动学

逐连杆比较 MDH 与 MuJoCo 的位姿。第 7 帧位于 `link7`；若研究法兰或夹爪，需要相应修改末端模型。


In [ ]:
robot = Robot.from_mdh("franka_panda_arm", PANDA_MDH_PARMS)
robot.print_dh_table()
print(f"dof={robot.dof}, convention={robot.convention}")

model = strip_to_rigid_body(mujoco.MjModel.from_xml_path(str(panda_xml_path())))
ref = MujocoReference(model, PANDA_MDH_BODY_NAMES)

q_fk = np.array([0.1, -0.4, 0.2, -1.8, 0.3, 1.2, 0.5])
poses = ref.body_poses(q_fk)
print("body      pos_err      rot_err")
for i in range(robot.dof):
    Ti = robot.fk_numpy(q_fk, link=i)
    Tm = poses[i]
    pe = np.linalg.norm(Ti[:3, 3] - Tm[:3, 3])
    re = np.linalg.norm(Ti[:3, :3] - Tm[:3, :3])
    print(f"link{i+1:<4d} {pe:.3e}  {re:.3e}")


## 2. 数值回归矩阵

`calc_base_parms(numeric=True)` 用数值逆动力学构造回归矩阵，再通过采样 QR 提取基参数。下面检查 `Hb @ π_b` 是否重现 MuJoCo 的刚体逆动力学力矩。


In [ ]:
t0 = time.time()
dyn = robot.dynamics
dyn.calc_base_parms(samples=800, numeric=True)
dyn.gen_base_regressor()
Hb_numeric = dyn.Hb_func
print(f"数值基参数就绪：{dyn.n_dynparms} → {dyn.n_base}，用时 {time.time()-t0:.1f}s")

pi_true = np.asarray(ref.dynparms(), float)
Pb = np.asarray(dyn.Pb, float)
Kd = np.asarray(dyn.Kd, float)
Pd = np.asarray(dyn.Pd, float)
# 基参数组合关系：π_b = (Pbᵀ + Kd Pdᵀ) π
pi_b_true = (Pb.T + Kd @ Pd.T) @ pi_true
print("pi_b_true shape:", pi_b_true.shape)

q0 = q_fk
dq0 = np.array([0.2, -0.1, 0.3, -0.2, 0.1, -0.15, 0.05])
ddq0 = np.array([-0.3, 0.4, -0.2, 0.5, -0.1, 0.2, -0.25])
tau_mj = ref.inverse_dynamics(q0, dq0, ddq0)
tau_hat = Hb_numeric(q0, dq0, ddq0) @ pi_b_true
print("||Hb_num @ pi_b - mj_inverse|| =", np.linalg.norm(tau_hat - tau_mj))


## 3. 可选：符号回归对照

本单元默认 `RUN_SYMBOLIC = False`。改为 `True` 后会生成完整符号回归矩阵并进行 `lambdify`，7 轴计算可能需要数分钟和较多内存。后续辨识始终使用第 2 节的 `Hb_numeric`，无需运行本节。


In [ ]:
RUN_SYMBOLIC = False

if RUN_SYMBOLIC:
    # 独立实例保留数值模型的基参数状态。
    robot_sym = Robot.from_mdh("franka_panda_arm_sym", PANDA_MDH_PARMS)
    dyn_sym = robot_sym.dynamics

    t0 = time.time()
    print("生成符号回归矩阵 H …", flush=True)
    dyn_sym.gen_regressor()
    print(f"  gen_regressor: {time.time()-t0:.1f}s", flush=True)

    t0 = time.time()
    dyn_sym.calc_base_parms(numeric=False)  # 在符号 H 的 lambdify 上采样 QR
    print(f"  calc_base_parms: {time.time()-t0:.1f}s，基参数 {dyn_sym.n_base}", flush=True)

    t0 = time.time()
    Hb_sym_expr = dyn_sym.gen_base_regressor()
    print(f"  gen_base_regressor: {time.time()-t0:.1f}s", flush=True)
    print("前 2 个基参数:")
    for expr in list(dyn_sym.baseparms)[:2]:
        display(expr)

    s = robot_sym.symbols
    Hb_lambdified = sympy.lambdify(
        list(s.q) + list(s.dq) + list(s.ddq), Hb_sym_expr, "numpy"
    )


    def Hb_symbolic(q, dq, ddq):
        return np.asarray(Hb_lambdified(*(list(q) + list(dq) + list(ddq))), float)


    # 与数值路径对齐（同一构型）
    Hs = Hb_symbolic(q0, dq0, ddq0)
    Hn = Hb_numeric(q0, dq0, ddq0)
    print("||Hb_sym - Hb_num||_F =", np.linalg.norm(Hs - Hn))
    print("||Hb_sym @ pi_b - mj_inverse|| =", np.linalg.norm(Hs @ pi_b_true - tau_mj))
else:
    print("跳过符号对照，后续使用 Hb_numeric。")


## 4. 激励、采样与辨识

搜索 12 条候选轨迹，按关节限位缩放后比较回归矩阵条件数。力矩由理想逆动力学生成，分别加入 `0`、`0.05`、`0.2 N·m` 白噪声；本节不模拟闭环跟踪或传感器延迟。


In [ ]:
Hb_func = Hb_numeric  # 跑完上一单元后也可换成 Hb_symbolic

lo = model.jnt_range[: robot.dof, 0].copy()
hi = model.jnt_range[: robot.dof, 1].copy()
for i in range(robot.dof):
    print(f"joint{i+1}: [{lo[i]:7.4f}, {hi[i]:7.4f}]")

rng = np.random.default_rng(0)
t0 = time.time()
traj, cond = search_excitation(
    Hb_func,
    robot.dof,
    rng,
    trials=12,
    vel_scale=0.8,
    limits=(lo, hi),
    refine=False,  # notebook 演示关掉 SLSQP；需要可改 True（要 scipy）
)
print(f"激励 cond(W)={cond:.3g}，搜索用时 {time.time()-t0:.1f}s")

data = sample_trajectory(traj, ref.inverse_dynamics, rate=80)
result = identify(Hb_func, data)
err = parameter_error(result.parms, pi_b_true)
print(result.summary())
print(f"相对误差 {err['rel_norm']:.3e}，最大分量误差 {err['abs_max']:.3e}")

for std in (0.0, 0.05, 0.2):
    noisy = data if std == 0 else add_noise(data, np.random.default_rng(1), tau_std=std)
    res = identify(Hb_func, noisy)
    e = parameter_error(res.parms, pi_b_true)["rel_norm"]
    print(f"tau_std={std:<4}  参数相对误差 {e:.3e}  残差 RMS {res.residual_rms:.3e}")


## 5. 用基参数分离 M / c / g

在特定速度、加速度下求值 `Hb`，得到惯性矩阵、科氏/离心力矩和重力项，再与 MuJoCo 力矩比较。该分解不需要恢复逐连杆惯性参数。


In [ ]:
decomposer = DynamicsDecomposer(Hb_func, result.parms, robot.dof)
terms = decomposer.at(q0, dq0)
tau_split = terms.torque(ddq0)
tau_id = Hb_func(q0, dq0, ddq0) @ result.parms
tau_mj = ref.inverse_dynamics(q0, dq0, ddq0)

print("||tau_split - tau_mj|| =", np.linalg.norm(tau_split - tau_mj))
print("||Hb@pi_hat - tau_mj|| =", np.linalg.norm(tau_id - tau_mj))
print("M", terms.M.shape, "条件数", np.linalg.cond(terms.M), "对称误差", terms.symmetry_error())


进一步比较激励轨迹见 [excitation.ipynb](excitation.ipynb)；完整符号导出与可选惯性参数恢复见 [modeling.ipynb](modeling.ipynb)。查看模型姿态可用 [visualization.ipynb](visualization.ipynb)。
